In [5]:
pip install modelscope

Looking in indexes: https://mirrors.cloud.aliyuncs.com/pypi/simple

[notice] A new release of pip is available: 23.3.2 -> 25.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [6]:
from modelscope import snapshot_download
model_dir = snapshot_download("Qwen/Qwen2.5-0.5B-Instruct")

2026-01-09 15:01:14,289 - modelscope - INFO - Target directory already exists, skipping creation.


In [3]:
pip install torch

Looking in indexes: https://mirrors.cloud.aliyuncs.com/pypi/simple

[notice] A new release of pip is available: 23.3.2 -> 25.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [7]:
import torch
print(torch.__version__)

2.9.1+cu128


In [8]:
pip install pdfplumber

Looking in indexes: https://mirrors.cloud.aliyuncs.com/pypi/simple
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 7.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 170.1 MB/s eta 0:00:0000:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 205.7 MB/s eta 0:00:00

[notice] A new release of pip is available: 23.3.2 -> 25.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [9]:
pip install sentence-transformers

Looking in indexes: https://mirrors.cloud.aliyuncs.com/pypi/simple

[notice] A new release of pip is available: 23.3.2 -> 25.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [10]:
pip install faiss-cpu

Looking in indexes: https://mirrors.cloud.aliyuncs.com/pypi/simple

[notice] A new release of pip is available: 23.3.2 -> 25.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [7]:
pip install transformers


Looking in indexes: https://mirrors.cloud.aliyuncs.com/pypi/simple

[notice] A new release of pip is available: 23.3.2 -> 25.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [11]:
pip install accelerate

Looking in indexes: https://mirrors.cloud.aliyuncs.com/pypi/simple

[notice] A new release of pip is available: 23.3.2 -> 25.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [12]:
import os
os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'
import pdfplumber

# 假设你已上传 your_document.pdf 到当前目录
pdf_path = "21.pdf"
text = ""

with pdfplumber.open(pdf_path) as pdf:
    for page in pdf.pages:
        page_text = page.extract_text()
        if page_text:
            text += page_text + "\n"

print(f"共提取 {len(text)} 个字符")

共提取 2571 个字符


In [13]:
def split_text(text, chunk_size=500, overlap=50):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        chunks.append(chunk)
        start = end - overlap  # 滑动窗口避免截断句子
    return chunks

chunks = split_text(text, chunk_size=500, overlap=50)
print(f"共切分为 {len(chunks)} 个文本块")

共切分为 6 个文本块


In [17]:
import os
os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'

In [18]:
# 更健壮的版本（推荐）
import os
os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'

from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

# 确保 chunks 不为空
if not chunks:
    raise ValueError("❌ chunks 为空！请先从 PDF 提取并分块文本。")

print(f"正在为 {len(chunks)} 个文本块生成嵌入向量...")

model_emb = SentenceTransformer('BAAI/bge-small-zh-v1.5')
embeddings = model_emb.encode(
    chunks,
    normalize_embeddings=True,
    show_progress_bar=True  # 显示进度条
)

embedding_dim = embeddings.shape[1]
index = faiss.IndexFlatIP(embedding_dim)
index.add(np.array(embeddings))

print(f"✅ 向量库构建完成！维度: {embedding_dim}, 文本块数: {len(chunks)}")

正在为 6 个文本块生成嵌入向量...


'(MaxRetryError("HTTPSConnectionPool(host='huggingface.co', port=443): Max retries exceeded with url: /BAAI/bge-small-zh-v1.5/resolve/main/modules.json (Caused by NewConnectionError('<urllib3.connection.HTTPSConnection object at 0x7f027cd2b810>: Failed to establish a new connection: [Errno 101] Network is unreachable'))"), '(Request ID: abf02f72-35fe-4569-bce9-2bc5b73d84c4)')' thrown while requesting HEAD https://huggingface.co/BAAI/bge-small-zh-v1.5/resolve/main/./modules.json
Retrying in 1s [Retry 1/5].
'(MaxRetryError("HTTPSConnectionPool(host='huggingface.co', port=443): Max retries exceeded with url: /BAAI/bge-small-zh-v1.5/resolve/main/modules.json (Caused by NewConnectionError('<urllib3.connection.HTTPSConnection object at 0x7f027cd7ee50>: Failed to establish a new connection: [Errno 101] Network is unreachable'))"), '(Request ID: 1f10ead0-bf3c-4bc2-bf7b-1e1a86e213fa)')' thrown while requesting HEAD https://huggingface.co/BAAI/bge-small-zh-v1.5/resolve/main/./modules.json
Retryi

KeyboardInterrupt: 

In [ ]:
def retrieve(query, top_k=3):
    query_vec = model_emb.encode([query], normalize_embeddings=True)
    scores, indices = index.search(np.array(query_vec), top_k)
    results = [chunks[i] for i in indices[0]]
    return results

In [45]:
from modelscope import AutoModelForCausalLM, AutoTokenizer

# 使用 Qwen2-7B-Instruct（需注意显存，若显存不足可换 1.8B 或 0.5B）
model_name = "Qwen/Qwen2.5-0.5B-Instruct"  # 轻量版更适配 Notebook

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    trust_remote_code=True,
     # 若支持
)
model.eval()

2026-01-05 13:20:58,570 - modelscope - INFO - Target directory already exists, skipping creation.


2026-01-05 13:21:00,134 - modelscope - INFO - Target directory already exists, skipping creation.


Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 896)
    (layers): ModuleList(
      (0-23): 24 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=896, out_features=896, bias=True)
          (k_proj): Linear(in_features=896, out_features=128, bias=True)
          (v_proj): Linear(in_features=896, out_features=128, bias=True)
          (o_proj): Linear(in_features=896, out_features=896, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
          (up_proj): Linear(in_features=896, out_features=4864, bias=False)
          (down_proj): Linear(in_features=4864, out_features=896, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((896,), eps=1e-06)
    (rotary_emb): Qwen2

In [46]:
def rag_qa_detailed(question, top_k=4):  # 增加 top_k 获取更多上下文
    # 1. 检索（可适当增加 top_k）
    retrieved_docs = retrieve(question, top_k=top_k)
    context = "\n".join(f"[片段{i+1}] {doc}" for i, doc in enumerate(retrieved_docs))
    
    # 2. 构造更强指令的 prompt
    messages = [{
        "role": "user",
        "content": (
            "你是一个专业、细致、知识渊博的 AI 助手，请严格根据以下参考资料回答问题。\n"
            "要求：\n"
            "- 回答应尽可能详细、全面，包含关键细节；\n"
            "- 如果涉及流程、功能或步骤，请分点说明（使用 1. 2. 3. 或 •）；\n"
            "- 不要编造信息，所有内容必须基于参考资料；\n"
            "- 如果参考资料中没有相关信息，请明确回答：“根据提供的资料无法回答该问题。”\n\n"
            f"参考资料：\n{context}\n\n"
            f"问题：{question}\n\n"
            "请给出详细回答："
        )
    }]
    
    # 3. 编码
    encoded = tokenizer.apply_chat_template(
        messages,
        return_tensors="pt",
        tokenize=True,
        return_dict=True
    )
    input_ids = encoded["input_ids"].to(model.device)
    attention_mask = encoded["attention_mask"].to(model.device)
    
    # 4. 生成（启用采样 + 调整参数以支持详细输出）
    with torch.no_grad():
        outputs = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=896,          # ← 增大！允许长回答
            do_sample=True,              # ← 启用采样，提升流畅性和丰富度
            temperature=0.6,             # ← 适度增加随机性（0.5~0.8）
            top_p=0.9,                   # ← nucleus sampling
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
            repetition_penalty=1.1       # ← 轻微惩罚重复，避免啰嗦
        )
    
    # 5. 解码
    response = tokenizer.decode(outputs[0][input_ids.shape[1]:], skip_special_tokens=True)
    return response.strip(), retrieved_docs

In [3]:
question = "为什么在高并发服务器中推荐使用线程池而不是为每个请求创建新线程？"

# 使用增强版函数
answer, sources = rag_qa_detailed(question)

print("❓ 问题：", question)
print("\n💡 详细回答：\n", answer)
#print("\n📄 参考片段（前200字）：")
# for i, src in enumerate(sources, 1):
#     print(f"[{i}] {src[:200]}...")

NameError: name 'rag_qa_detailed' is not defined

In [2]:
import gradio as gr
def qa_interface(question: str):
    if not question or not question.strip():
        return "请输入一个问题。", []
    
    try:
        answer, sources = rag_qa_detailed(question.strip())
        
        # 格式化参考片段：截断 + 添加省略号
        formatted_sources = []
        for i, src in enumerate(sources, 1):
            snippet = src[:250].strip()
            if len(src) > 250:
                snippet += "..."
            formatted_sources.append(f"【参考 {i}】\n{snippet}")
        
        # 将 sources 转为 Markdown 列表（Gradio 支持）
        sources_md = "\n\n".join(formatted_sources) if formatted_sources else "无参考片段。"
        
        return answer, sources_md
    
    except Exception as e:
        return f"❌ 检索出错：{str(e)}", ""


# 构建 Gradio 界面
with gr.Blocks(title="RAG 技术问答助手") as demo:
    gr.Markdown("## 💬 高并发系统 RAG 问答助手")
    gr.Markdown("基于技术文档的智能问答，支持操作系统、网络、并发编程等领域")
    
    with gr.Row():
        with gr.Column(scale=2):
            question_input = gr.Textbox(
                label="❓ 请输入你的技术问题",
                placeholder="例如：为什么在高并发服务器中推荐使用线程池而不是为每个请求创建新线程？",
                lines=3
            )
            submit_btn = gr.Button("🔍 获取答案", variant="primary")
            
            gr.Examples(
                examples=[
                    "为什么在高并发服务器中推荐使用线程池而不是为每个请求创建新线程？",
                    "epoll 和 select 的区别是什么？",
                    "什么是 C10K 问题？如何解决？",
                    "Redis 为什么是单线程的？"
                ],
                inputs=question_input
            )
        
        with gr.Column(scale=3):
            answer_output = gr.Textbox(
                label="💡 详细回答",
                interactive=False,
                lines=8
            )
            sources_output = gr.Markdown(label="📄 参考片段（来自技术文档）")

    # 绑定事件
    submit_btn.click(
        fn=qa_interface,
        inputs=question_input,
        outputs=[answer_output, sources_output]
    )

if __name__ == "__main__":
    demo.launch(server_name="0.0.0.0", server_port=7860, share=True)

/usr/local/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


OSError: Cannot find empty port in range: 7860-7860. You can specify a different port by setting the GRADIO_SERVER_PORT environment variable or passing the `server_port` parameter to `launch()`.